# 01. LangChain tool call


> 랭체인에서도 tool 사용 가능

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("하이~")])

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a5a1892700', 'id': 'chatcmpl-DuuNg3aJKYJ4QKvSB1DsKREWadLlR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02a0-02e2-7a61-883d-7a8e7baa9028-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
from langchain_core.tools import tool  #ChatOpenAI는 tool 모듈이 따로 있다. 더 편하다.

In [ ]:
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록. 이러면 일반 함수로는 못쓴다.
def get_current_time(timezone: str, location: str) -> str:
    ### 단순한 주석이 아니라 랭체인에 이 함수의 기능과 입력값, 사용 방법을 알려주는 문서화 문자열. 기존의 desc을 함수 내부에 '''으로 주석으로 넣어주는것.
    """ 현재 시각을 반환하는 함수 

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [ ]:
get_current_time(timezone= 'Asia', location= 'Seoul') #일반함수로 안돌아가네..

UnknownTimeZoneError: 'Asia'

In [4]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]  #이 안에 넣을 함수는 @tool을 한 함수여야 한다.
tool_dict = {"get_current_time": get_current_time,}

In [5]:
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [6]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("서울은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

In [11]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-DuuTanVwVQL8SEGLDTPpXi0mlwizA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02a5-97fb-71f2-acc9-7790d2f988b1-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_tR4Hd7XR652fLpKqoVw0oBcH', 'type': 'tool_call'}

In [12]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
  'id': 'call_tR4Hd7XR652fLpKqoVw0oBcH',
  'type': 'tool_call'}]

In [13]:
tool_call = response.tool_calls[0]
tool_call["name"]

'get_current_time'

In [14]:
tool_dict[tool_call["name"]]

StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수 \n\n   Args:\n       timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n       location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x0000026090A40E00>)

In [ ]:
response.tool_calls[0]  #답변으로 나온 tool_call 정보

{'name': 'get_current_time',
 'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
 'id': 'call_tR4Hd7XR652fLpKqoVw0oBcH',
 'type': 'tool_call'}

In [16]:
tool_dict[tool_call["name"]].invoke(tool_call)  #tool_dict의 'get_current_time' 함수를 가져와 'tool_call'이라는 질의의 답을 가져온다.
#tool_call의 내용물은 response.tool_calls[0]이다. 앞서 사용된 툴을 사용해 나온 답변 정보가 들어있다.
#답변에 사용된 함수 정보가 들어있는데 왜 앞에 또 한번 언급하지??

Asia/Seoul (서울) 현재시각 2026-06-26 15:45:22 


ToolMessage(content='Asia/Seoul (서울) 현재시각 2026-06-26 15:45:22 ', name='get_current_time', tool_call_id='call_tR4Hd7XR652fLpKqoVw0oBcH')

In [ ]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

In [ ]:
llm_with_tools.invoke(messages)

In [ ]:
messages.append(llm_with_tools.invoke(messages))

In [ ]:
messages

### 주가 예제

In [33]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [34]:
messages=[]
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
messages.append(response)

In [35]:
response.tool_calls

[{'name': 'get_yf_stock_history',
  'args': {'ticker': 'TSLA', 'period': '1mo'},
  'id': 'call_40TbbNADD870H8JpeA2pQcL9',
  'type': 'tool_call'}]

In [36]:
messages

[HumanMessage(content='테슬라는 한달 전에 비해 주가가 올랐나 내렸나?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 203, 'total_tokens': 226, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DuupLkYPDtlz4Mhg1KeTXxwfd3g6k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02ba-2fbf-7681-a661-7c86679aa87b-0', tool_calls=[{'name': 'get_yf_stock_history', 'args': {'ticker': 'TSLA', 'period': '1mo'}, 'id': 'call_40TbbNADD870H8JpeA2pQcL9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 23, 'total_tokens': 

In [31]:
!pip install tabulate

In [37]:
#### 이어서 자연어로 모델이 답변하는 것 추가 실습


for tool_call in response.tool_calls:
    out=tool_dict[tool_call['name']].invoke(tool_call)
    messages.append(out)
llm_with_tools.invoke(messages)


AIMessage(content='한 달 전 테슬라(TSLA)의 주가는 약 430.26달러로 시작했으며, 현재 주가는 375.12달러입니다. 따라서 한 달 동안 테슬라의 주가는 내렸습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 1559, 'total_tokens': 1609, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1536}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-Duupa7m9gdcwshZwlgrQFBZO77AiY', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02ba-6adf-7502-a9e8-d6d04ebafd6b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1559, 'output_tokens': 50, 'total_tokens': 1609, 'input_token_details': {'audio': 0, 'cache_read': 1536}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 실습 1. pdf_summary.py

pdf_summary 하는 함수 tool로 등록하여 사용해보기

+ summarize_text 도 langchain 사용하는 것으로 변경

In [ ]:
from langchain_core.messages import HumanMessage
import os 

pdf_path = os.path.join(os.getcwd(),"samples/videoanomaly.pdf")

messages = [
    HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?"),
]  

In [42]:
from pdf_summary_langchain_sbg import summarize_pdf

In [43]:
tool_list = [summarize_pdf]
tool_dict = {'summarize_pdf':summarize_pdf}

In [44]:
from langchain_openai import ChatOpenAI

llm_with_tools=llm.bind_tools(tool_list)

In [45]:
response=llm_with_tools.invoke(messages)

In [46]:
messages.append(response)

In [47]:
for tool_call in response.tool_calls:
    out=tool_dict[tool_call['name']].invoke(tool_call)

In [48]:
messages.append(out)

In [49]:
llm_with_tools.invoke(messages)

AIMessage(content='이 문서의 저자는 Qianzi Yu, Kai Zhu, Yang Cao, Yu Kang입니다. 이들은 중국 과학기술대학교 및 헌페이 종합 국가 과학 센터의 연구원들로, 인공지능 및 비디오 분석 분야에서 활발히 연구하고 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 549, 'total_tokens': 613, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42e6cfce1b', 'id': 'chatcmpl-DuvRVVPfyLOBX3tbQZMMpAxS117PC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02de-47d4-7c83-b6a7-7678e04fb7d0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 549, 'output_tokens': 64, 'total_tokens': 613, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})